## Sel 1: Import Pustaka

In [ ]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

import mlflow
import dagshub

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import xgboost as xgb

import optuna

import joblib
import contextlib
from tqdm.auto import tqdm

import warnings
warnings.filterwarnings('ignore')

## Sel 2: Memuat & Pra-pemrosesan Data (Resampling Harian)

In [ ]:
# Pastikan path ini sesuai dengan posisi .env milikmu
load_dotenv('../.env', override=True)

# Ambil URL MLflow utuh 
db_url_mlflow = os.getenv("DATABASE_URL_DIRECT")

if not db_url_mlflow:
    raise ValueError("DATABASE_URL_DIRECT tidak ditemukan di .env!")

# Buat URL polos khusus untuk Pandas & SQLAlchemy
db_url_pandas = db_url_mlflow.split("?")[0]

print("Membuka jembatan ke Supabase...")
# Gunakan URL polos untuk engine
engine = create_engine(db_url_pandas)

# ==========================================
# LANGKAH 1: KUERI SQL JATAH RATA
# ==========================================
RENTANG_WAKTU = "1 year"

query_sql = f"""
    SELECT 
        waktu_aktual, 
        id_wilayah, 
        pm25, pm10, so2, co, no2, ozon
    FROM public.data_historis
    WHERE waktu_aktual >= NOW() - INTERVAL '{RENTANG_WAKTU}';
"""

ukuran_chunk = 50000
chunks = []

with engine.connect() as conn:
    conn.execute(text("SET statement_timeout = 0"))
    conn.commit() # Wajib commit agar pengaturannya tersimpan di sesi ini
    
    stream_conn = conn.execution_options(stream_results=True)
    
    for i, chunk in enumerate(pd.read_sql(text(query_sql), stream_conn, chunksize=ukuran_chunk)):
        print(f"✅ Berhasil menyedot batch ke-{i+1} (hingga {(i+1) * ukuran_chunk} baris...)")
        chunks.append(chunk)

df = pd.concat(chunks, ignore_index=True)

kolom_waktu = 'waktu_aktual'
kolom_kota = 'id_wilayah'  
polutan = ['pm25', 'pm10', 'so2', 'co', 'no2', 'ozon'] 

# PRA-PEMROSESAN & RESAMPLE (PERUBAHAN KE JAM)
df[kolom_waktu] = pd.to_datetime(df[kolom_waktu])
df[kolom_waktu] = df[kolom_waktu].dt.tz_localize(None)
df.set_index(kolom_waktu, inplace=True)

# PENTING: Resample 'H' (Hourly/Jam), bukan 'D' (Daily/Harian)
df_hourly = df.groupby(kolom_kota)[polutan].resample('H').mean().reset_index()

# ==========================================
# LANGKAH 2: INTERPOLASI 
# ==========================================
print("Menambal data sensor yang bolong dengan interpolasi...")

df_hourly = df_hourly.groupby(kolom_kota, group_keys=False).apply(
    lambda group: group.interpolate(method='linear')
).reset_index(drop=True)

df_hourly = df_hourly.bfill().ffill()

print(f"Selesai! Bentuk data akhir (PER JAM): {df_hourly.shape}")
display(df_hourly.head())

## Sel 3: Rekayasa Fitur

In [ ]:
def create_advanced_features_24h(df_kota, n_lags=24, n_targets=24):
    # Pastikan data selalu diurutkan dari terlama ke terbaru sebelum digeser!
    df_temp = df_kota.sort_values(kolom_waktu).copy()
    
    # FITUR TEMPORAL
    df_temp['Bulan'] = df_temp[kolom_waktu].dt.month
    df_temp['Jam'] = df_temp[kolom_waktu].dt.hour
    df_temp['Is_Weekend'] = df_temp[kolom_waktu].dt.dayofweek.isin([5, 6]).astype(int)

    for p in polutan:
        # FITUR HISTORY: Lihat 24 jam ke belakang
        for i in range(1, n_lags + 1):
            df_temp[f'{p}_H-{i}'] = df_temp[p].shift(i)
            
        # FITUR ROLLING: 72 Jam Terakhir (3 Hari)
        df_temp[f'{p}_RollMean_72'] = df_temp[p].rolling(window=72).mean()
        df_temp[f'{p}_RollMax_72'] = df_temp[p].rolling(window=72).max()
        
        # TARGET 24 JAM KE DEPAN (Multi-Output)
        for h in range(1, n_targets + 1):
            df_temp[f'TARGET_{p}_T+{h}'] = df_temp[p].shift(-h)
            
    return df_temp

print("Meracik fitur 24 Jam...")
df_model = df_hourly.groupby(kolom_kota, group_keys=False).apply(create_advanced_features_24h)
df_model = df_model.dropna().reset_index(drop=True)

df_model[kolom_kota] = df_model[kolom_kota].astype(str)
df_model = pd.get_dummies(df_model, columns=[kolom_kota])

kolom_target = [col for col in df_model.columns if 'TARGET_' in col]
y = df_model[kolom_target]

kolom_bawaan = [kolom_waktu, 'kategori_ispu']
X = df_model.drop(columns=kolom_target + kolom_bawaan, errors='ignore')

print(f"Dimensi X (Soal AI)   : {X.shape}")
print(f"Dimensi y (Target 24J): {y.shape}")

# Sanity Check
assert X.isna().sum().sum() == 0, "Ada NaN di data X!"
assert y.isna().sum().sum() == 0, "Ada NaN di data y!"

display(X.head())
display(y.head())

## Sel 4: Splitting & Tracking Baseline Model MLFLOW

### Splitting

In [ ]:
# 1. Splitting Data (Time Series Split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

print("Mengecilkan ukuran data di RAM (Downcasting ke Float32)...")
X_train = X_train.astype('float32')
X_test = X_test.astype('float32')
y_train = y_train.astype('float32')
y_test = y_test.astype('float32')

# ==========================================
# TRANSFORMASI LOGARITMIK (ANTI-MINUS)
# ==========================================
print("Menerapkan Transformasi Logaritmik (Log1p) pada Target untuk mencegah prediksi negatif...")

# y = log(y + 1). Ini memaksa rentang data menjadi lebih rapat dan mencegah output negatif.
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

print("Data target telah dikompresi ke skala logaritmik!")
display(y_train_log.head(3))

### Tracking MLFlow DagsHub

In [ ]:
# ==========================================
# TRACKING MLFLOW DENGAN DAGSHUB
# ==========================================
# 2. Ambil Konfigurasi DagsHub dari .env
repo_owner = os.getenv("DAGSHUB_REPO_OWNER")
repo_name = os.getenv("DAGSHUB_REPO_NAME")

if not repo_owner or not repo_name:
    raise ValueError("Identitas DAGSHUB_REPO_OWNER atau DAGSHUB_REPO_NAME belum di-set di file .env!")

print(f"Connecting to DagsHub MLflow Server ({repo_owner}/{repo_name})...")
dagshub.init(repo_owner=repo_owner, repo_name=repo_name, mlflow=True)

## Sel 5 : Hyperparameter Tuning XGBoost MLFlow

In [ ]:
print("Memulai Hyperparameter Tuning XGBoost Multioutput (Dengan Proteksi Logaritmik)...")

# 1. Set Eksperimen Tahap 2 (Nama Baru untuk Native Multi)
mlflow.set_experiment("XGB_Optuna_Native_MultiOutput")

print("\nMenyiapkan Data untuk Akselerasi GPU (Konversi ke Numpy C-Contiguous)...")
X_train_np = np.ascontiguousarray(X_train.values.astype('float32'))
X_test_np = np.ascontiguousarray(X_test.values.astype('float32'))

model_terbaik_per_polutan = {}
total_mae, total_rmse = 0, 0

# --- MENGELOMPOKKAN 144 TARGET MENJADI 6 GRUP ---
daftar_polutan = ['PM25', 'PM10', 'SO2', 'CO', 'NO2', 'OZON'] 
kelompok_target = {}

for polutan in daftar_polutan:
    kolom_terkait = [col for col in y_train_log.columns if polutan in col.upper()]
    if kolom_terkait:
        kelompok_target[polutan] = kolom_terkait
    else:
        print(f"PERINGATAN: Kolom untuk polutan {polutan} tidak ditemukan!")

# ==========================================
# 2. FUNGSI OPTUNA DENGAN RUANG PENCARIAN DIPERLUAS
# ==========================================
def objective(trial, X_ctx, y_ctx):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500, step=50),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-5, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-5, 10.0, log=True)
    }
    
    model = xgb.XGBRegressor(
        **params, 
        random_state=42, 
        tree_method="hist", 
        device="cuda",
        multi_strategy="multi_output_tree",
        n_jobs=1
    )
    
    score = cross_val_score(model, X_ctx, y_ctx, scoring='neg_mean_absolute_error', cv=3).mean()
    return score

# ==========================================
# 3. PROSES TUNING 6 POLUTAN (ZIP & UNZIP)
# ==========================================
for nama_polutan, list_kolom_24_jam in kelompok_target.items():
    print(f"\n======================================================")
    print(f"--> Mendidik AI Spesialis Khusus: {nama_polutan} (Native Multi-Output)")
    print(f"======================================================")
    
    y_train_grup_log = y_train_log[list_kolom_24_jam]
    y_train_grup_log_np = np.ascontiguousarray(y_train_grup_log.values.astype('float32'))
    
    y_test_grup_asli = y_test[list_kolom_24_jam]
    
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(direction='maximize')
    study.optimize(lambda trial: objective(trial, X_train_np, y_train_grup_log_np), n_trials=20, show_progress_bar=True)
    
    best_params = study.best_params
    print(f"[+] Kombinasi Terbaik Ditemukan:\n{best_params}")
    
    otak_spesialis = xgb.XGBRegressor(
        **best_params, 
        random_state=42, 
        tree_method="hist", 
        device="cuda",
        multi_strategy="multi_output_tree",
        n_jobs=1
    )
    
    otak_spesialis.fit(X_train_np, y_train_grup_log_np)
    model_terbaik_per_polutan[nama_polutan] = otak_spesialis
    
    # [FASE UNZIP]: Prediksi dan Kembalikan ke Wujud Asli (Microgram)
    y_pred_grup_log = otak_spesialis.predict(X_test_np)
    y_pred_grup_asli = np.expm1(y_pred_grup_log)
    
    # Menghitung MAE dan RMSE Keseluruhan
    mae_kombinasi = mean_absolute_error(y_test_grup_asli, y_pred_grup_asli)
    rmse_kombinasi = np.sqrt(mean_squared_error(y_test_grup_asli, y_pred_grup_asli))
    
    # --- IMPLEMENTASI R-SQUARED (R2) ---
    r2_kombinasi = r2_score(y_test_grup_asli, y_pred_grup_asli) # R2 Keseluruhan 24 jam
    r2_per_jam = r2_score(y_test_grup_asli, y_pred_grup_asli, multioutput='raw_values') # R2 spesifik tiap jam
    
    rata_rata_asli = y_test_grup_asli.values.mean()
    persentase_error = (mae_kombinasi / rata_rata_asli) * 100 if rata_rata_asli > 0 else 0
    
    total_mae += mae_kombinasi
    total_rmse += rmse_kombinasi
    
    print(f"  - MAE (Meleset Rata-rata) : {mae_kombinasi:.2f}")
    print(f"  - RMSE Rata-rata 24 Jam   : {rmse_kombinasi:.2f}")
    print(f"  - R-Squared (R2) Score    : {r2_kombinasi:.4f}") # Menampilkan skor R2 di terminal
    print(f"  - Error (%)               : {persentase_error:.2f}%")
    
    # PENCATATAN DAGSHUB
    with mlflow.start_run(run_name=f"Optuna_XGB_Native_{nama_polutan}_Tegar_Log"):
        mlflow.log_params(best_params)
        mlflow.log_metric("test_mae_kombinasi", mae_kombinasi)
        mlflow.log_metric("test_rmse_kombinasi", rmse_kombinasi)
        mlflow.log_metric("test_r2_kombinasi", r2_kombinasi) # Mencatat R2 Keseluruhan
        mlflow.log_metric("error_percentage", persentase_error)
        
        # Mencatat R2 per jam ke MLflow (menggantikan RMSE per jam)
        for jam in range(len(list_kolom_24_jam)):
            mlflow.log_metric(f"r2_jam_ke_{jam+1}", r2_per_jam[jam])

print("\n--- 6 MODEL SELESAI DILATIH (NATIVE MULTI-OUTPUT + LOG TRANSFORM) ---")
if len(kelompok_target) > 0:
    print(f"Rata-rata MAE Keseluruhan ({len(kelompok_target)} Model) : {total_mae/len(kelompok_target):.2f}")

Memulai Hyperparameter Tuning XGBoost (Strategi Pisahkan Otak)...

🚀 Mendidik AI Spesialis Khusus: PM25


Tuning PM25:   0%|          | 0/60 [00:00<?, ?fit/s]

Kombinasi Terbaik:
{'subsample': 1.0, 'n_estimators': 300, 'max_depth': 9, 'learning_rate': 0.2, 'colsample_bytree': 0.9}
  - Rata-rata asli : 5.28
  - MAE (Meleset)  : 1.56
  - RMSE           : 3.10
  - Error (%)      : 29.64%

🚀 Mendidik AI Spesialis Khusus: PM10


Tuning PM10:   0%|          | 0/60 [00:00<?, ?fit/s]

Kombinasi Terbaik:
{'subsample': 1.0, 'n_estimators': 300, 'max_depth': 9, 'learning_rate': 0.2, 'colsample_bytree': 0.9}
  - Rata-rata asli : 10.85
  - MAE (Meleset)  : 2.58
  - RMSE           : 4.05
  - Error (%)      : 23.80%

🚀 Mendidik AI Spesialis Khusus: SO2


...

Kombinasi Terbaik:
{'subsample': 0.9, 'n_estimators': 400, 'max_depth': 7, 'learning_rate': 0.05, 'colsample_bytree': 0.8}
  - Rata-rata asli : 119.16
  - MAE (Meleset)  : 21.74
  - RMSE           : 41.98
  - Error (%)      : 18.24%

🚀 Mendidik AI Spesialis Khusus: NO2


Tuning NO2:   0%|          | 0/60 [00:00<?, ?fit/s]

Kombinasi Terbaik:
{'subsample': 0.9, 'n_estimators': 400, 'max_depth': 7, 'learning_rate': 0.05, 'colsample_bytree': 0.8}
  - Rata-rata asli : 0.34
  - MAE (Meleset)  : 0.21
  - RMSE           : 0.47
  - Error (%)      : 61.49%

🚀 Mendidik AI Spesialis Khusus: OZON


Tuning OZON:   0%|          | 0/60 [00:00<?, ?fit/s]

Kombinasi Terbaik:
{'subsample': 1.0, 'n_estimators': 300, 'max_depth': 9, 'learning_rate': 0.2, 'colsample_bytree': 0.9}
  - Rata-rata asli : 39.57
  - MAE (Meleset)  : 3.63
  - RMSE           : 5.71
  - Error (%)      : 9.18%

--- 🏁 SEMUA 6 AI SPESIALIS SELESAI DILATIH ---
Rata-rata MAE Keseluruhan : 4.96


## Sel 6: Audit Keadilan Per Wilayah

In [ ]:
# =====================================================================
# 1. TABEL STATISTIK AKURASI KESELURUHAN POLUTAN (SETELAH TUNING)
# =====================================================================
print("--- TABEL STATISTIK AKURASI TIAP POLUTAN (SETELAH TUNING) ---")

statistik_polutan = []

# Konversi X_test ke Numpy agar XGBoost GPU berjalan maksimal (tanpa warning)
X_test_np = np.ascontiguousarray(X_test.values.astype('float32'))

# Looping ke semua model spesialis polutan yang sudah di-training
for polutan, model in model_terbaik_per_polutan.items():
    
    # 1. Ambil 24 kolom target yang asli milik polutan ini
    kolom_target_24jam = [col for col in y_test.columns if polutan in col.upper()]
    y_true = y_test[kolom_target_24jam]
    
    # 2. Lakukan prediksi 24 jam (Output berupa Numpy Array)
    y_pred = model.predict(X_test_np)

    # 3. Kalkulasi Metrik Evaluasi Keseluruhan (Gunakan .values agar jadi skalar)
    rata_rata_asli = y_true.values.mean()
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    # Cegah error pembagian dengan nol
    persentase_error = (mae / rata_rata_asli) * 100 if rata_rata_asli != 0 else 0

    # Masukkan hasil ke dalam list
    statistik_polutan.append({
        "Polutan": polutan.upper(),
        "Rata-rata Asli": f"{rata_rata_asli:.2f}",
        "MAE (Meleset)": f"{mae:.2f}",
        "RMSE": f"{rmse:.2f}",
        "Error (%)": f"{persentase_error:.2f}%"
    })

# Ubah list menjadi DataFrame Pandas agar tampilannya berbentuk tabel cantik
df_statistik = pd.DataFrame(statistik_polutan)
display(df_statistik)

print("\n" + "="*60 + "\n")

# =====================================================================
# 2. AUDIT KEADILAN AI PER WILAYAH (Khusus PM2.5)
# =====================================================================
print("--- AUDIT KEADILAN AI PER WILAYAH (Khusus PM2.5 24-Jam) ---")

# Ambil spesialis PM2.5 secara eksplisit
kunci_pm25 = 'PM25' 
if kunci_pm25 in model_terbaik_per_polutan:
    model_pm25 = model_terbaik_per_polutan[kunci_pm25]

    # Prediksi menggunakan format Numpy array
    y_pred_pm25 = model_pm25.predict(X_test_np)
    
    # Ambil 24 kolom target PM2.5
    kolom_pm25 = [col for col in y_test.columns if 'PM25' in col.upper()]
    y_test_pm25 = y_test[kolom_pm25]

    # Definisikan nama awalan One-Hot Encoding dari Supabase (Asumsi: id_wilayah)
    kolom_kota = 'id_wilayah'
    kolom_wilayah = [col for col in X_test.columns if col.startswith(f'{kolom_kota}_')]

    # Lakukan loop untuk mengecek akurasi di tiap kota
    for wil in kolom_wilayah:
        # Filter (masking) berdasarkan Pandas
        mask = X_test[wil] == 1

        if mask.sum() > 0: # Pastikan kotanya ada di data test
            # Slice Pandas DataFrame
            y_test_wilayah = y_test_pm25[mask]
            
            # Slice Numpy Array wajib menggunakan mask.values
            y_pred_wilayah = y_pred_pm25[mask.values]

            # Hitung Error (Meleset berapa angka untuk tebakan 24 jam di kota ini)
            mae_wilayah = mean_absolute_error(y_test_wilayah, y_pred_wilayah)

            # Bersihkan nama untuk diprint (ambil ID angkanya saja)
            id_asli = wil.replace(f'{kolom_kota}_', '')

            print(f"ID Wilayah {id_asli:<3} | MAE Prediksi PM2.5: {mae_wilayah:.2f}")
else:
    print("Model PM25 tidak ditemukan di dalam dictionary memori!")

--- 📊 TABEL STATISTIK AKURASI TIAP POLUTAN (SETELAH TUNING) ---


,Polutan,Rata-rata Asli,MAE (Meleset),RMSE,Error (%)
0,TARGET_PM25_BESOK,5.28,1.56,3.10,29.64%
1,TARGET_PM10_BESOK,10.85,2.58,4.05,23.80%
2,TARGET_SO2_BESOK,0.17,0.06,0.13,34.48%
3,TARGET_CO_BESOK,119.16,21.74,41.98,18.24%
4,TARGET_NO2_BESOK,0.34,0.21,0.47,61.49%
5,TARGET_OZON_BESOK,39.57,3.63,5.71,9.18%




--- ⚖️ AUDIT KEADILAN AI PER WILAYAH (Khusus PM2.5) ---
ID Wilayah 31  | MAE PM2.5: 1.06
ID Wilayah 32  | MAE PM2.5: 1.46
ID Wilayah 33  | MAE PM2.5: 1.64
ID Wilayah 34  | MAE PM2.5: 1.13
ID Wilayah 35  | MAE PM2.5: 1.42
ID Wilayah 36  | MAE PM2.5: 1.77
ID Wilayah 37  | MAE PM2.5: 2.16
ID Wilayah 38  | MAE PM2.5: 1.67


## Sel 7: Ekspor Model (Menyimpan Hasil)

In [ ]:
print("Memulai proses verifikasi dan ekspor model...")

# 1. SANITY CHECK: Pastikan ke-6 model polutan benar-benar ada di memori
daftar_polutan_wajib = ['PM25', 'PM10', 'SO2', 'CO', 'NO2', 'OZON']
polutan_hilang = [p for p in daftar_polutan_wajib if p not in model_terbaik_per_polutan]

if polutan_hilang:
    print(f"Batal Ekspor! PERINGATAN KRITIS: Otak model untuk {polutan_hilang} tidak ditemukan di memori!")
    
else:
    print("Pengecekan aman: 6 model spesialis lengkap di memori.")
    
    # 2. Simpan
    nama_file = "xgb_optuna_multioutput_tegar.pkl"

    # 3. Membungkus KE-ENAM model spesialis dan daftar urutan fitur
    paket_model = {
        'dict_model_spesialis': model_terbaik_per_polutan,
        'fitur': list(X_train.columns)
    }

    # 4. Simpan langsung ke kandang DVC
    joblib.dump(paket_model, nama_file)

    print(f"Model pintar telah diekspor sebagai '{nama_file}'")
    print(f"Total fitur cuaca/historis yang direkam: {len(paket_model['fitur'])} kolom")

📦 Selesai! Model pintar telah diekspor sebagai 'xgb_ispu_jatim_multi_otak.pkl'
